# Naver 뉴스 본문 수집 (BS4 Colab용) — 언론사+기간 모드

URL 수집 노트북에서 만든 `링크_{press}_*.json` 파일을 읽어 `requests`와 `BeautifulSoup`으로 기사 제목, 본문, 날짜, 카테고리를 수집한다. 통신3사 트랙과 동일하게 **기간 단위 통합 CSV** 1개를 만든다 (입력 JSON 1개 → 출력 CSV 1개). 네이버 뉴스 본문이 초기 HTML에 포함되어 있어 Selenium보다 가볍게 수집할 수 있다.

- 입력: `data/링크_{press}_{YYMMDD}_{YYMMDD}.json` (기간 통합)
- 출력: `data/본문_bs4_{press}_{YYMMDD}_{YYMMDD}.csv`
- 보조 출력: 중간 재개용 체크포인트 JSON, 재실패 URL JSON
- 특징: 기간 단위 통합 수집, 중간 재개, 오류 URL 1회 재시도, 스포츠/연예 리다이렉트 자동 분기, 완료 파일 건너뛰기


In [1]:
# Colab 환경 세팅 — requests, BeautifulSoup 설치
# !pip install -q requests beautifulsoup4 pandas


In [2]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Drive 안의 프로젝트 폴더로 이동
import os
PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring'
os.chdir(PROJECT_DIR)
print(f'현재 작업 폴더: {os.getcwd()}')


현재 작업 폴더: /content/drive/MyDrive/Text-data-Analysis_26-Spring


In [4]:
import json
import os
import random
import re
import time
import unicodedata
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

# 로컬/Colab 비교를 위해 User-Agent 고정
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'

# 담당 언론사와 수집 기간 지정
# URL 수집 노트북과 동일한 press_ranges를 그대로 사용 (oid는 본문 수집에선 사용 안 함)
# press 하나당 start_date ~ end_date 통합 1개 CSV로 저장 (통신3사 트랙과 동일 패턴)
# 날짜 형식: 'YYYY.MM.DD'
press_ranges = [
    # 지상파 (이미 수집됨 — 본문_bs4_*.csv가 있으면 SKIP_COMPLETED로 건너뜀)
    {'press': 'SBS', 'oid': '055', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': 'KBS', 'oid': '056', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': 'MBC', 'oid': '214', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    # 경제
    {'press': '한국경제', 'oid': '015', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': '매일경제', 'oid': '009', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    # 정치색
    {'press': '한겨레', 'oid': '028', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': '조선일보', 'oid': '023', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    # 통신·보도(YTN/연합뉴스)는 김은수가 수집 → 본문_bs4_YTN_*, 본문_bs4_연합뉴스_* 파일로 SAVE_DIR에 넣으면 통합 셀이 자동 포함
]


# 기간 단위 작업 목록 생성 — press_ranges 한 항목당 jobs 1개
# URL 수집 노트북과 같은 시그니처(press/start_date/end_date)를 본문 수집에 그대로 전달
def build_period_jobs(press_ranges):
    jobs = []
    for item in press_ranges:
        # 'YYYY.MM.DD' 문자열을 datetime으로 파싱 — 기간 유효성 검사용
        start = datetime.strptime(item['start_date'], '%Y.%m.%d')
        end = datetime.strptime(item['end_date'], '%Y.%m.%d')
        # 시작 일자가 끝 일자보다 늦으면 작업 범위가 잘못된 것이므로 즉시 중단
        if start > end:
            raise ValueError(f"시작 일자가 끝 일자보다 늦습니다: {item}")
        # oid는 본문 수집에선 안 쓰니까 제외 (링크 파일명은 press로만 매칭)
        jobs.append({
            'press': item['press'],
            'start_date': item['start_date'],
            'end_date': item['end_date'],
        })
    return jobs


# 변수 정의 (날짜 형식: 'YYYY.MM.DD')
# 생성된 jobs는 다음 셀에서 순서대로 실행
jobs = build_period_jobs(press_ranges)

print(f'총 작업 수: {len(jobs)}')
for job in jobs:
    print(job)

# 셀 3을 건너뛰고 실행해도 기본 프로젝트 경로를 사용할 수 있게 보완
try:
    PROJECT_DIR
except NameError:
    PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring'

# 저장할 폴더 지정 — 링크 파일, 체크포인트, 본문 CSV, 실패 목록이 모두 이 폴더에 저장
SAVE_DIR = Path(PROJECT_DIR) / 'data' / 'news'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'저장 위치: {SAVE_DIR}')

# requests 세션 생성 — 같은 User-Agent와 언어 설정을 반복 요청에 적용
# Selenium 대신 requests를 쓰는 이유: 네이버 뉴스 본문은 초기 HTML에 포함돼 JS 렌더링 불필요
session = requests.Session()
session.headers.update({
    'User-Agent': USER_AGENT,
    'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
})
print(f'User-Agent: {USER_AGENT}')


총 작업 수: 3
{'press': 'SBS', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
{'press': 'KBS', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
{'press': 'MBC', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}
저장 위치: /content/drive/MyDrive/Text-data-Analysis_26-Spring/data/news
User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36


In [5]:
# 서버 부담을 줄이기 위해 기사/job 사이에 짧은 랜덤 대기
# BS4는 Selenium보다 빠르므로 기사 간 대기는 좀 더 짧게 설정
ARTICLE_PAUSE_RANGE_SEC = (0.4, 1.2)
JOB_PAUSE_RANGE_SEC = (8, 20)

# 중간에 끊겨도 이어서 수집할 수 있도록 일정 건수마다 체크포인트 저장
# BS4는 빠르므로 Selenium(100)보다 큰 간격(200) 사용
CHECKPOINT_INTERVAL = 200
REQUEST_TIMEOUT_SEC = 10
SKIP_COMPLETED = True

# 일반 뉴스 URL이 리다이렉트되는 서브포털 도메인 매핑
# 키: 호스트 substring 검사용 / 값: (카테고리 prefix, URL path 첫 segment 추출 정규식)
# 도메인이 추가되면 여기에만 등록하면 분기 함수가 자동으로 처리
REDIRECT_DOMAINS = {
    'sports.naver.com': ('스포츠', r'https://m\.sports\.naver\.com/([^/]+)/article/'),
    'entertain.naver.com': ('연예', r'https://m\.entertain\.naver\.com/([^/]+)/article/'),
}


# 랜덤 대기 후 로그 출력
def polite_sleep(label, pause_range):
    pause_sec = random.uniform(*pause_range)
    print(f"{label} {pause_sec:.1f}초 대기")
    time.sleep(pause_sec)


# 파일명에 사용할 YYMMDD_YYMMDD 형식 기간 문자열 생성 (통신3사 트랙과 동일 시그니처)
# 예: 2026.05.01 ~ 2026.05.07 -> '260501_260507'
def make_period_suffix(start_date, end_date):
    return f"{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}"


# CSS selector 결과가 없으면 빈 문자열 반환 — 누락 필드 판정은 호출부에서 일괄 처리
def get_text_or_empty(soup, selector):
    element = soup.select_one(selector)
    return element.get_text(strip=True) if element else ''


# 본문 안의 줄바꿈/연속 공백을 하나의 공백으로 정리 (CSV/분석 단계 일관성 위해)
def normalize_body_text(text):
    return re.sub(r'\s+', ' ', text).strip()


# 한글 표시 시각을 일반 뉴스의 data-date-time과 같은 포맷으로 변환
# 예: '2026.05.05. 오전 7:10' -> '2026-05-05 07:10:00'
def parse_korean_datetime(text):
    m = re.match(r'(\d{4})\.(\d{2})\.(\d{2})\.\s*(오전|오후)\s*(\d{1,2}):(\d{2})', text)
    if not m:
        return ''
    y, mo, d, ampm, h, mi = m.groups()
    h = int(h)
    # 12시간제 → 24시간제 변환 (오전 12시 = 00시, 오후 12시 = 12시 그대로)
    if ampm == '오후' and h != 12:
        h += 12
    elif ampm == '오전' and h == 12:
        h = 0
    return f'{y}-{mo}-{d} {h:02d}:{mi}:00'


# 네이버 서브포털(스포츠/연예)은 일반 뉴스와 HTML 구조가 달라 별 셀렉터 사용
# 두 도메인 모두 Next.js 기반이라 og:title / div._article_content / em.date 셀렉터를 공유
def extract_redirected_article_bs4(response, original_link):
    soup = BeautifulSoup(response.text, 'html.parser')

    # 제목 추출하기 — meta og:title 사용 (Next.js CSS 모듈 hash가 빌드마다 바뀌어 더 안정적)
    title_meta = soup.find('meta', property='og:title')
    title = title_meta.get('content', '').strip() if title_meta else ''

    # 본문 추출하기 — div._article_content (언더스코어 prefix는 hash에 의존하지 않음)
    body_el = soup.select_one('div._article_content')
    body = normalize_body_text(body_el.get_text(strip=True)) if body_el else ''

    # 날짜 추출하기 — em.date 첫 번째(입력일). 두 번째는 수정일이라 무시
    em = soup.select_one('em.date')
    pubdate = parse_korean_datetime(em.get_text(strip=True)) if em else ''

    # 카테고리 추출하기 — 도메인 매핑 dict에서 prefix를 찾고 URL path 첫 segment를 붙임
    # 예: m.sports.naver.com/golf/article/... -> '스포츠/golf'
    category = '기타'
    for domain, (prefix, pattern) in REDIRECT_DOMAINS.items():
        if domain in response.url:
            cat_match = re.match(pattern, response.url)
            category = f'{prefix}/{cat_match.group(1)}' if cat_match else prefix
            break

    # 제목/본문/날짜 중 하나라도 없으면 실패로 기록하고 재시도 대상에 포함
    if not title or not body or not pubdate:
        raise ValueError(f'리다이렉트 기사: title={bool(title)}, body={bool(body)}, pubdate={bool(pubdate)}')

    # 호출부에서 원래 URL을 키로 쓰므로 link 필드는 원본 URL로 유지
    return {
        'link': original_link,
        'pubdate': pubdate,
        'category': category,
        'title': title,
        'body': body,
    }


# 기사 한 건에서 title/body/pubdate/category 추출 (BS4 버전)
# BS4로 본문 추출 — Selenium 대비 약 10배 빠름 (네이버 뉴스는 초기 HTML에 본문 포함)
def extract_article_bs4(link, session=session):
    # 실제 네이버 뉴스 웹페이지 HTML 요청 (리다이렉트는 default로 따라감)
    response = session.get(link, timeout=REQUEST_TIMEOUT_SEC)
    response.raise_for_status()

    # 리다이렉트 감지 — 일반 뉴스 URL이 스포츠/연예 서브포털로 빠지면 별 추출기로 분기
    if any(domain in response.url for domain in REDIRECT_DOMAINS):
        return extract_redirected_article_bs4(response, link)

    soup = BeautifulSoup(response.text, 'html.parser')

    # 제목 추출하기
    title = get_text_or_empty(soup, '.media_end_head_headline')

    # 본문 추출하기 — 공백 정리해 단일 문자열로
    body = get_text_or_empty(soup, '#newsct_article')
    body = normalize_body_text(body)

    # 날짜 추출하기 — 사람이 읽는 라벨이 아니라 data-date-time 속성값(ISO 포맷) 사용
    pubdate_element = soup.select_one('span.media_end_head_info_datestamp_time._ARTICLE_DATE_TIME')
    pubdate = pubdate_element.get('data-date-time', '') if pubdate_element else ''

    # 카테고리 추출하기 — 상단 탭 중 현재 활성화된(aria-selected="true") 항목
    category = get_text_or_empty(soup, 'a.Nitem_link[aria-selected="true"] span.Nitem_link_menu')

    # 제목/본문/날짜 중 하나라도 없으면 실패로 기록하고 재시도 대상에 포함
    if not title or not body or not pubdate:
        raise ValueError('title/body/pubdate 중 일부를 추출하지 못함')

    return {
        'link': link,
        'pubdate': pubdate,
        'category': category,
        'title': title,
        'body': body,
    }


# 한 언론사의 기간 통합 링크 JSON을 읽어 본문을 수집하고 통합 CSV로 저장
# 체크포인트 기반 재개 + 오류 자동 1회 재시도 + 재실패 URL JSON 저장
# Drive 동기화로 한글 파일명이 NFD로 풀려 들어와도 찾도록, NFC로 대조해 실제 파일 경로를 돌려준다
# (일치하는 기존 파일이 없으면 NFC 이름의 새 경로를 반환 — 새로 저장할 때 사용)
def resolve_nfc_path(save_dir, filename):
    target = unicodedata.normalize('NFC', filename)
    direct = save_dir / filename
    if direct.exists():
        return direct
    for cand in save_dir.iterdir():
        if unicodedata.normalize('NFC', cand.name) == target:
            return cand
    return save_dir / target


def collect_bodies_bs4(press, start_date, end_date, save_dir=SAVE_DIR):
    # 파일명 키로 쓸 기간 접미사 (예: 260501_260507)
    period = make_period_suffix(start_date, end_date)
    # 입력/출력 경로 — 한글 파일명 NFC/NFD 차이를 흡수해 기존 파일을 찾는다 (없으면 NFC 새 경로)
    links_path = resolve_nfc_path(save_dir, f"링크_{press}_{period}.json")
    checkpoint_path = resolve_nfc_path(save_dir, f"체크포인트_본문_bs4_{press}_{period}.json")
    csv_save_path = resolve_nfc_path(save_dir, f"본문_bs4_{press}_{period}.csv")

    # 최종 파일이 이미 있으면 같은 기간은 건너뜀 (재실행 시 idempotent)
    if SKIP_COMPLETED and csv_save_path.exists():
        print()
        print(f"=== {press} / {start_date} ~ {end_date} 이미 완료됨, 건너뜀 ===")
        print(f"기존 파일: {csv_save_path}")
        return csv_save_path

    # 링크 파일 불러오기
    if links_path.exists():
        with links_path.open('r', encoding='utf-8') as f:
            naver_news_links = json.load(f)
        # 대략적인 예상 시간 출력 — 실제 시간은 네트워크 상태/timeout/재시도에 따라 달라질 수 있음
        avg_pause_sec = sum(ARTICLE_PAUSE_RANGE_SEC) / 2
        est_sec_per_article = avg_pause_sec + 0.7
        est_min = len(naver_news_links) * est_sec_per_article / 60
        print()
        print(f"=== {press} / {start_date} ~ {end_date} BS4 본문 수집 시작 ===")
        print(f'링크 {len(naver_news_links)}개 불러옴: {links_path}')
        print(f'예상 소요 시간: 약 {est_min:.0f}분')
    else:
        raise FileNotFoundError(f'링크 파일 없음 — 언론사_네이버뉴스_url_수집_colab.ipynb를 먼저 실행하세요\n경로: {links_path}')

    # 이전에 중단된 작업이 있으면 이어받기 — next_i 인덱스 다음부터 시작
    if checkpoint_path.exists():
        with checkpoint_path.open('r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        # JSON은 dict 키를 문자열로 저장하므로 int로 다시 변환
        all_results = {int(k): v for k, v in checkpoint.get('all_results', {}).items()}
        err_idx = checkpoint.get('err_idx', [])
        i = checkpoint.get('next_i', 0)
        print(f'체크포인트 발견 — {i}번째부터 이어서 시작 (이미 수집: {len(all_results)}건)')
    else:
        all_results = dict()
        i = 0
        err_idx = []
        print('새로 시작')

    # 추출한 링크에 직접 방문하여 크롤링 진행
    for link in naver_news_links[i:]:
        try:
            all_results[i] = extract_article_bs4(link)

            # 진행 상황 확인용 코드
            print(f'[{i+1} / {len(naver_news_links)}] \t {(i+1)/len(naver_news_links)*100:.2f}% \t error: {len(err_idx)}')

            i += 1

            # 중간저장 — N건마다 체크포인트 갱신해 중간 중단에도 진행 보존
            if i % CHECKPOINT_INTERVAL == 0:
                with checkpoint_path.open('w', encoding='utf-8') as f:
                    json.dump({'all_results': all_results, 'err_idx': err_idx, 'next_i': i}, f, ensure_ascii=False, indent=2)
                print(f'체크포인트 저장 — {i}건 완료')

            # 봇 탐지 방지를 위해 다음 기사 요청 전 랜덤한 시간을 기다림
            polite_sleep('다음 기사 전', ARTICLE_PAUSE_RANGE_SEC)

        except Exception as exc:
            print(f'오류 발생 — index {i}: {exc!r}')
            # 실패한 index는 err_idx에 저장한 뒤 마지막에 한 번 더 재시도
            err_idx.append(i)
            i += 1
            # 오류 직후에도 체크포인트 즉시 갱신해 err_idx 누락 방지
            with checkpoint_path.open('w', encoding='utf-8') as f:
                json.dump({'all_results': all_results, 'err_idx': err_idx, 'next_i': i}, f, ensure_ascii=False, indent=2)

    # 1차 오류 자동 재시도 — 일시적 네트워크 문제나 응답 지연이었던 경우를 한 번 더 시도
    if err_idx:
        print()
        print(f'오류 {len(err_idx)}건 재시도 시작...')
        re_err_idx = []

        for retry_i in err_idx:
            try:
                link = naver_news_links[retry_i]
                all_results[retry_i] = extract_article_bs4(link)
                print(f'재시도 성공 — index {retry_i}')
                polite_sleep('다음 재시도 전', ARTICLE_PAUSE_RANGE_SEC)
            except Exception as exc:
                print(f'재시도 실패 — index {retry_i}: {exc!r}')
                re_err_idx.append(retry_i)

        # 재시도 후에도 실패한 인덱스만 남김 — 본문_bs4_재실패_*.json으로 저장 대상
        err_idx = re_err_idx
        print(f'재시도 완료 — 재실패: {len(err_idx)}건')

    # 수집한 정보들을 dataframe으로 변환
    df = pd.DataFrame(all_results).T

    if df.empty:
        raise ValueError('수집된 본문 데이터가 없습니다.')

    # 수집한 기사들 중 중복인 경우 이를 제거
    df_no_duplicates = df.drop_duplicates().reset_index(drop=True)

    # 오래된 순부터 수집했으나 혹시 모를 상황을 방지하기 위해 pubdate를 datetime으로 변환 후 정렬
    df_no_duplicates['pubdate'] = pd.to_datetime(df_no_duplicates['pubdate'], errors='coerce')
    df_sorted = df_no_duplicates.sort_values(by='pubdate')

    # 수집한 정보들을 csv로 저장 (Google Drive에 저장)
    df_sorted.to_csv(csv_save_path, index=False, encoding='utf-8-sig')
    print(f'저장 완료: {csv_save_path}')

    if err_idx:
        # 재시도 후에도 실패한 URL은 별도 JSON으로 저장 — Selenium 재시도 노트북에서 다시 시도 가능
        failed_path = save_dir / f"본문_bs4_재실패_{press}_{period}.json"
        with failed_path.open('w', encoding='utf-8') as f:
            json.dump({'err_idx': err_idx, 'links': [naver_news_links[x] for x in err_idx]}, f, ensure_ascii=False, indent=2)
        print(f'재실패 목록 저장: {failed_path}')
    elif checkpoint_path.exists():
        # 재실패가 없을 때만 체크포인트 삭제 — 실패가 남아있으면 디버깅용으로 보존
        checkpoint_path.unlink()

    print(f'본문 수집 완료 — 총 {len(df_sorted)}건 / 오류 {len(err_idx)}건')
    return csv_save_path


# 생성된 jobs를 순서대로 실행
# 한 작업이 실패해도 실패 목록에 기록하고 다음 작업으로 넘어감
results = []
failures = []
for index, job in enumerate(jobs, start=1):
    print()
    print(f"[{index}/{len(jobs)}] 작업 실행: {job}")
    try:
        # job 딕셔너리의 press/start_date/end_date를 collect_bodies_bs4 인자로 전달
        results.append(collect_bodies_bs4(**job))
    except Exception as exc:
        # 한 언론사에서 오류가 나도 전체 작업이 멈추지 않도록 실패 정보만 저장
        failures.append({'job': job, 'error': repr(exc)})
        print(f"작업 실패, 다음 작업으로 넘어감: {exc!r}")
    finally:
        if index < len(jobs):
            # 다음 언론사 job으로 넘어가기 전 대기
            polite_sleep('다음 작업 전', JOB_PAUSE_RANGE_SEC)

# 실패한 작업이 있으면 나중에 다시 돌릴 수 있게 파일로 저장
if failures:
    failures_path = SAVE_DIR / '본문_bs4_수집실패목록_naver.json'
    with failures_path.open('w', encoding='utf-8') as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)
    print()
    print(f"실패 작업 {len(failures)}개 저장: {failures_path}")

print()
print('전체 작업 완료')
print(f'성공/건너뜀: {len(results)}개, 실패: {len(failures)}개')
for result_path in results:
    print(result_path)



[1/3] 작업 실행: {'press': 'SBS', 'start_date': '2026.05.05', 'end_date': '2026.05.11'}

=== SBS / 2026.05.05 ~ 2026.05.11 BS4 본문 수집 시작 ===
링크 1339개 불러옴: /content/drive/MyDrive/Text-data-Analysis_26-Spring/data/news/링크_SBS_260505_260511.json
예상 소요 시간: 약 33분
새로 시작
[1 / 1339] 	 0.07% 	 error: 0
다음 기사 전 0.5초 대기
[2 / 1339] 	 0.15% 	 error: 0
다음 기사 전 0.9초 대기
[3 / 1339] 	 0.22% 	 error: 0
다음 기사 전 0.8초 대기
[4 / 1339] 	 0.30% 	 error: 0
다음 기사 전 0.9초 대기
[5 / 1339] 	 0.37% 	 error: 0
다음 기사 전 0.8초 대기
[6 / 1339] 	 0.45% 	 error: 0
다음 기사 전 0.5초 대기
[7 / 1339] 	 0.52% 	 error: 0
다음 기사 전 0.7초 대기
[8 / 1339] 	 0.60% 	 error: 0
다음 기사 전 1.1초 대기
[9 / 1339] 	 0.67% 	 error: 0
다음 기사 전 1.0초 대기
[10 / 1339] 	 0.75% 	 error: 0
다음 기사 전 0.9초 대기
[11 / 1339] 	 0.82% 	 error: 0
다음 기사 전 1.1초 대기
[12 / 1339] 	 0.90% 	 error: 0
다음 기사 전 0.5초 대기
[13 / 1339] 	 0.97% 	 error: 0
다음 기사 전 0.4초 대기
[14 / 1339] 	 1.05% 	 error: 0
다음 기사 전 0.9초 대기
[15 / 1339] 	 1.12% 	 error: 0
다음 기사 전 0.8초 대기
[16 / 1339] 	 1.19% 	 error: 0
다음 기사 전 1.0초

## press별 본문 CSV 통합 (수집 직후 자동 실행)

위 셀이 만든 `본문_bs4_{press}_{기간}.csv`들을 모두 읽어 `press` 컬럼을 붙여
하나의 `통합_본문_bs4_언론사_{기간}.csv`로 저장한다. 이게 전처리/분석 단계의 입력 1개가 된다.

- 입력/출력 모두 위에서 정의한 `SAVE_DIR` 안 (수집물이 저장된 곳과 동일)
- `_direct`(방송사 직접 트랙) 파일은 통합에서 제외
- press마다 oid가 달라 매체 간 link 충돌은 없지만, 안전하게 link 기준 한 번 더 중복 제거

In [ ]:
# === press별 본문 CSV 통합 (+ media_group 부착) ===
import re
import unicodedata

# press -> media_group 매핑 (분석 비교 단위). 새 매체를 수집/반입하면 여기만 추가
# 팀원 파일도 본문_bs4_{press}_{기간}.csv 형식으로 이름만 맞춰 SAVE_DIR에 두면 자동 포함됨
MEDIA_GROUP_MAP = {
    'KBS': '지상파', 'MBC': '지상파', 'SBS': '지상파',
    'YTN': '통신·보도', '연합뉴스': '통신·보도',
    '한국경제': '경제', '매일경제': '경제',
    '조선일보': '정치색', '한겨레': '정치색',
}
EXCLUDE_DIRECT = True  # 방송사 직접 트랙(_direct) 결과물은 통합에서 제외


def _nfc(name):
    # Drive 동기화로 한글 파일명이 NFD로 들어올 수 있어 매칭 전 NFC로 통일
    return unicodedata.normalize('NFC', name)

# SAVE_DIR의 본문_bs4_{press}_{YYMMDD}_{YYMMDD}.csv를 모두 모음
# 현재 press_ranges 기간만 통합 (폴더에 다른 기간 파일이 남아 섞이는 것 방지)
target_periods = {make_period_suffix(r['start_date'], r['end_date']) for r in press_ranges}
print(f'통합 대상 기간: {sorted(target_periods)}')

infos = []
for path in SAVE_DIR.iterdir():
    if not path.is_file():
        continue
    m = re.match(r'^본문_bs4_(.+)_(\d{6})_(\d{6})\.csv$', _nfc(path.name))
    if not m:
        continue
    press, s, e = m.groups()
    if EXCLUDE_DIRECT and press.endswith('_direct'):
        print(f'제외(_direct): {_nfc(path.name)}')
        continue
    if f'{s}_{e}' not in target_periods:
        print(f'기간 불일치 제외: {_nfc(path.name)} (대상 {sorted(target_periods)})')
        continue
    infos.append((press, s, e, path))

infos.sort(key=lambda x: (x[0], x[1]))
if not infos:
    raise FileNotFoundError(f'통합할 본문_bs4_*.csv가 없습니다: {SAVE_DIR}')

frames = []
for press, s, e, path in infos:
    d = pd.read_csv(path, encoding='utf-8-sig')
    d['press'] = press                       # 파일명에서 떼어낸 press를 행마다 부착
    d['source_period'] = f'{s}_{e}'
    d['source_file'] = _nfc(path.name)
    frames.append(d)
    print(f'  {press} {s}_{e}: {len(d)}행')

merged = pd.concat(frames, ignore_index=True)

# media_group 부착 — 매핑에 없는 press는 미분류로 두고 경고 (수집 이름과 키를 맞추세요)
merged['media_group'] = merged['press'].map(MEDIA_GROUP_MAP)
unmapped = sorted(merged.loc[merged['media_group'].isna(), 'press'].unique())
if unmapped:
    print(f'\n[경고] media_group 매핑 없는 press: {unmapped} -> 미분류 (MEDIA_GROUP_MAP에 추가하세요)')
merged['media_group'] = merged['media_group'].fillna('미분류')

before = len(merged)
if 'link' in merged.columns:
    merged = merged.drop_duplicates(subset=['link'], keep='first').reset_index(drop=True)
if 'pubdate' in merged.columns:
    merged['pubdate'] = pd.to_datetime(merged['pubdate'], errors='coerce')
    merged = merged.sort_values(['media_group', 'press', 'pubdate']).reset_index(drop=True)

period_suffix = f"{min(s for _, s, _, _ in infos)}_{max(e for _, _, e, _ in infos)}"
combined_path = SAVE_DIR / f'통합_본문_bs4_언론사_{period_suffix}.csv'
merged.to_csv(combined_path, index=False, encoding='utf-8-sig')

print(f'\n통합 저장 완료: {combined_path}')
print(f'행 수: {before} -> {len(merged)} (중복 {before - len(merged)}건 제거)')
print('media_group별:', merged['media_group'].value_counts().to_dict())
print('press별:', merged['press'].value_counts().to_dict())